# Phase 5: Hyperparameter Tuning

**Predicting 30-Day Hospital Readmission for Diabetic Patients**

Two-stage tuning (RandomizedSearchCV -> GridSearchCV) for LR, RF, and XGBoost. SVM excluded due to 45-minute training time. Goal: maximize ROC-AUC while monitoring recall.

**Inputs:** Phase 3 data + Phase 4 baseline models and CV results
**Outputs:** Tuned models, final model selection -> Phase 6

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, pickle, warnings, time, os
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

COLORS = {
    'primary': '#1B2A4A', 'steel': '#4A7FB5', 'teal': '#2AA198',
    'orange': '#E67E22', 'red': '#E74C3C', 'green': '#2ECC71', 'gold': '#D4A017'
}

ARTIFACTS_DIR = '../artifacts'
os.makedirs('../figures', exist_ok=True)
os.makedirs('../models', exist_ok=True)
print('Environment ready.')

### Load Phase 3 + Phase 4 Artifacts

In [ ]:
# Load Phase 3 data
X_train = pd.read_parquet(f'{ARTIFACTS_DIR}/phase3_X_train.parquet')
X_test  = pd.read_parquet(f'{ARTIFACTS_DIR}/phase3_X_test.parquet')
y_train = pd.read_parquet(f'{ARTIFACTS_DIR}/phase3_y_train.parquet').squeeze()
y_test  = pd.read_parquet(f'{ARTIFACTS_DIR}/phase3_y_test.parquet').squeeze()
preprocessor = joblib.load(f'{ARTIFACTS_DIR}/phase3_preprocessor.joblib')

with open(f'{ARTIFACTS_DIR}/phase3_meta.pkl', 'rb') as f:
    p3_meta = pickle.load(f)

# Load Phase 4 baseline pipelines and results
models = {}
for name in ['Logistic Regression', 'Random Forest', 'XGBoost', 'SVM']:
    safe = name.lower().replace(' ', '_')
    fpath = f'{ARTIFACTS_DIR}/phase4_baseline_{safe}.joblib'
    if os.path.exists(fpath):
        models[name] = joblib.load(fpath)
        print(f'  Loaded: {name}')

with open(f'{ARTIFACTS_DIR}/phase4_cv_results.pkl', 'rb') as f:
    results = pickle.load(f)

with open(f'{ARTIFACTS_DIR}/phase4_meta.pkl', 'rb') as f:
    p4_meta = pickle.load(f)

best_model_name = p4_meta['best_model_name']

# Recreate CV and scoring objects
from sklearn.model_selection import StratifiedKFold, cross_validate
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['roc_auc', 'recall', 'precision', 'f1']

print(f'\nPhase 3 + 4 artifacts loaded')
print(f'  Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'  Baseline best: {best_model_name}')

### 5.1 Tuning Strategy

**Two-stage approach:**
1. **RandomizedSearchCV** (20-80 iterations) -- broad sweep across the parameter space to find the right neighborhood. Faster than exhaustive grid search because it samples randomly rather than evaluating every combination.
2. **GridSearchCV** -- narrow, exhaustive search around the best parameters from Stage 1. Refines the final values within the region RandomizedSearch identified.

**Why two stages instead of one?** XGBoost alone has 6 tunable parameters. A full grid with 4 values each = 4,096 combinations x 5 folds = 20,480 fits. At ~6 seconds per model, that is 34 hours. RandomizedSearch with 80 iterations x 5 folds = 400 fits (~40 minutes), then GridSearch on a 3x3x3 narrowed space = 135 x 5 = 675 fits (~1 hour). Total: ~1.5 hours vs. 34 hours for the same quality result.

**Scoring:** ROC-AUC remains the primary optimization metric. Recall and F1 are monitored but not optimized directly -- threshold optimization in Phase 6 is the right lever for recall.

**Three models to tune:**
- Logistic Regression (baseline AUC: 0.698)
- Random Forest (baseline AUC: 0.696)
- XGBoost (baseline AUC: 0.679) -- included despite ranking 3rd because gradient boosting is the published benchmark for this dataset and often improves dramatically with tuning

### 5.2 Tune Logistic Regression

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
import time

lr_param_dist = {
    'classifier__C': [0.001, 0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__solver': ['saga'],  # saga supports both l1 and l2
}

print("Logistic Regression -- Stage 1: RandomizedSearchCV")
print(f"  Parameter space: {len(lr_param_dist['classifier__C'])} x {len(lr_param_dist['classifier__penalty'])} = {len(lr_param_dist['classifier__C']) * len(lr_param_dist['classifier__penalty'])} total combinations")

lr_random = RandomizedSearchCV(
    models['Logistic Regression'],
    param_distributions=lr_param_dist,
    n_iter=20,  # LR space is small enough for near-exhaustive
    cv=cv, scoring='roc_auc',
    random_state=42, n_jobs=-1, verbose=1
)

start = time.time()
lr_random.fit(X_train, y_train)
print(f"  Time: {time.time() - start:.1f}s")
print(f"  Best AUC: {lr_random.best_score_:.4f}")
print(f"  Best params: {lr_random.best_params_}")
print(f"  Baseline AUC was: 0.6982")
print(f"  Improvement: {lr_random.best_score_ - 0.6982:+.4f}")

In [ ]:
# Stage 2: GridSearchCV around best region
best_C = lr_random.best_params_['classifier__C']
best_penalty = lr_random.best_params_['classifier__penalty']

# Build narrow grid around best C
c_values = sorted(set([best_C * 0.5, best_C * 0.75, best_C, best_C * 1.5, best_C * 2.0]))

print("Logistic Regression -- Stage 2: GridSearchCV")
print(f"  Narrowing around C={best_C}, penalty={best_penalty}")
print(f"  C values: {c_values}")

lr_grid = GridSearchCV(
    models['Logistic Regression'],
    param_grid={
        'classifier__C': c_values,
        'classifier__penalty': [best_penalty],
        'classifier__solver': ['saga'],
    },
    cv=cv, scoring='roc_auc',
    n_jobs=-1, verbose=1
)

start = time.time()
lr_grid.fit(X_train, y_train)
print(f"  Time: {time.time() - start:.1f}s")
print(f"  Best AUC: {lr_grid.best_score_:.4f}")
print(f"  Best params: {lr_grid.best_params_}")
print(f"  Stage 1 AUC was: {lr_random.best_score_:.4f}")
print(f"  Stage 2 improvement: {lr_grid.best_score_ - lr_random.best_score_:+.4f}")

**Logistic Regression tuning results:** RandomizedSearchCV explored 20 combinations across C and penalty, finding the best configuration at C=0.1 with L1 penalty (AUC 0.6992, +0.001 over the 0.6982 baseline). GridSearchCV refined around that region but found no further improvement -- the RandomizedSearch had already identified the optimal neighborhood. Total tuning time was approximately 9 minutes (411s + 131s). The minimal gain was expected: LR has few hyperparameters, and the baseline with class_weight='balanced' and L2 regularization was already near-optimal. The switch from L2 to L1 provides slightly better feature selection via sparsity but did not meaningfully change discriminative performance.

### 5.3 Tune Random Forest

In [ ]:
rf_param_dist = {
    'classifier__n_estimators': [100, 200, 300, 500],
    'classifier__max_depth': [5, 10, 15, 20, 25, None],
    'classifier__min_samples_leaf': [20, 50, 100, 200],
    'classifier__max_features': ['sqrt', 'log2', 0.3, 0.5],
    'classifier__min_samples_split': [2, 5, 10, 20],
}

print("Random Forest -- Stage 1: RandomizedSearchCV")
total_combos = 4 * 6 * 4 * 4 * 4
print(f"  Total parameter space: {total_combos:,} combinations")
print(f"  Sampling 80 random combinations")

rf_random = RandomizedSearchCV(
    models['Random Forest'],
    param_distributions=rf_param_dist,
    n_iter=80,
    cv=cv, scoring='roc_auc',
    random_state=42, n_jobs=-1, verbose=1
)

start = time.time()
rf_random.fit(X_train, y_train)
print(f"  Time: {time.time() - start:.1f}s")
print(f"  Best AUC: {rf_random.best_score_:.4f}")
print(f"  Best params: {rf_random.best_params_}")
print(f"  Baseline AUC was: 0.6961")
print(f"  Improvement: {rf_random.best_score_ - 0.6961:+.4f}")

In [ ]:
# Stage 2: Narrow grid around best RF params
best_rf = rf_random.best_params_
best_depth = best_rf['classifier__max_depth']
best_leaf = best_rf['classifier__min_samples_leaf']
best_est = best_rf['classifier__n_estimators']

# Build narrow ranges
depth_vals = [best_depth] if best_depth is None else sorted(set([max(3, best_depth - 5), best_depth, best_depth + 5]))
leaf_vals = sorted(set([max(10, best_leaf - 20), best_leaf, best_leaf + 20]))

print("Random Forest -- Stage 2: GridSearchCV")
print(f"  Narrowing around depth={best_depth}, leaf={best_leaf}, n_estimators={best_est}")
print(f"  Depth values: {depth_vals}")
print(f"  Leaf values: {leaf_vals}")

rf_grid = GridSearchCV(
    models['Random Forest'],
    param_grid={
        'classifier__n_estimators': [best_est],
        'classifier__max_depth': depth_vals,
        'classifier__min_samples_leaf': leaf_vals,
        'classifier__max_features': [best_rf['classifier__max_features']],
    },
    cv=cv, scoring='roc_auc',
    n_jobs=-1, verbose=1
)

start = time.time()
rf_grid.fit(X_train, y_train)
print(f"  Time: {time.time() - start:.1f}s")
print(f"  Best AUC: {rf_grid.best_score_:.4f}")
print(f"  Best params: {rf_grid.best_params_}")
print(f"  Stage 1 AUC was: {rf_random.best_score_:.4f}")
print(f"  Stage 2 improvement: {rf_grid.best_score_ - rf_random.best_score_:+.4f}")

**Random Forest tuning results:** RandomizedSearchCV sampled 80 of 1,536 possible combinations across tree depth, leaf size, feature subsampling, and ensemble size, achieving AUC 0.6997 (best params: n_estimators=500, max_depth=25, min_samples_leaf=20, max_features=sqrt). GridSearchCV refined min_samples_leaf from 20 down to 10, pushing AUC to 0.7009 (+0.005 over the 0.6961 baseline). Total tuning time was approximately 10 minutes (530s + 76s). However, tuned recall collapsed from 0.574 (baseline) to 0.224 -- the tuned RF became better at overall discrimination but far more conservative in positive predictions, missing 78% of actual readmissions. This is a cautionary example of optimizing for AUC alone on imbalanced data: deeper trees with smaller leaf sizes improved the model's ability to rank patients but shifted the decision boundary toward predicting fewer positives.

### 5.4 Tune XGBoost

In [ ]:
xgb_param_dist = {
    'classifier__learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'classifier__max_depth': [3, 4, 5, 6, 7, 8],
    'classifier__n_estimators': [100, 200, 300, 500],
    'classifier__subsample': [0.7, 0.8, 0.9, 1.0],
    'classifier__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'classifier__min_child_weight': [1, 3, 5, 7],
}

print("XGBoost -- Stage 1: RandomizedSearchCV")
total_combos = 5 * 6 * 4 * 4 * 4 * 4
print(f"  Total parameter space: {total_combos:,} combinations")
print(f"  Sampling 80 random combinations")

xgb_random = RandomizedSearchCV(
    models['XGBoost'],
    param_distributions=xgb_param_dist,
    n_iter=80,
    cv=cv, scoring='roc_auc',
    random_state=42, n_jobs=-1, verbose=1
)

start = time.time()
xgb_random.fit(X_train, y_train)
print(f"  Time: {time.time() - start:.1f}s")
print(f"  Best AUC: {xgb_random.best_score_:.4f}")
print(f"  Best params: {xgb_random.best_params_}")
print(f"  Baseline AUC was: 0.6788")
print(f"  Improvement: {xgb_random.best_score_ - 0.6788:+.4f}")

In [ ]:
# Stage 2: Narrow grid around best XGBoost params
best_xgb = xgb_random.best_params_

lr_vals = sorted(set([
    max(0.005, best_xgb['classifier__learning_rate'] * 0.5),
    best_xgb['classifier__learning_rate'],
    best_xgb['classifier__learning_rate'] * 2.0
]))
depth_vals = sorted(set([
    max(2, best_xgb['classifier__max_depth'] - 1),
    best_xgb['classifier__max_depth'],
    best_xgb['classifier__max_depth'] + 1
]))

print("XGBoost -- Stage 2: GridSearchCV")
print(f"  Narrowing around learning_rate={best_xgb['classifier__learning_rate']}, max_depth={best_xgb['classifier__max_depth']}")
print(f"  Learning rate values: {lr_vals}")
print(f"  Depth values: {depth_vals}")

xgb_grid = GridSearchCV(
    models['XGBoost'],
    param_grid={
        'classifier__learning_rate': lr_vals,
        'classifier__max_depth': depth_vals,
        'classifier__n_estimators': [best_xgb['classifier__n_estimators']],
        'classifier__subsample': [best_xgb['classifier__subsample']],
        'classifier__colsample_bytree': [best_xgb['classifier__colsample_bytree']],
        'classifier__min_child_weight': [best_xgb['classifier__min_child_weight']],
    },
    cv=cv, scoring='roc_auc',
    n_jobs=-1, verbose=1
)

start = time.time()
xgb_grid.fit(X_train, y_train)
print(f"  Time: {time.time() - start:.1f}s")
print(f"  Best AUC: {xgb_grid.best_score_:.4f}")
print(f"  Best params: {xgb_grid.best_params_}")
print(f"  Stage 1 AUC was: {xgb_random.best_score_:.4f}")
print(f"  Stage 2 improvement: {xgb_grid.best_score_ - xgb_random.best_score_:+.4f}")

**XGBoost tuning results:** RandomizedSearchCV sampled 80 of 7,680 possible combinations and achieved AUC 0.7025 -- the largest improvement of any model (+0.024 over the 0.6788 baseline). Best parameters: learning_rate=0.03, max_depth=4, n_estimators=300, subsample=0.8, colsample_bytree=0.7, min_child_weight=1. GridSearchCV found no further improvement, confirming RandomizedSearch had already located the optimal region. Total tuning time was approximately 2 minutes (115s + 9s) -- by far the fastest of the three models. The key parameter changes tell a clear story: learning rate dropped from 0.1 to 0.03 (slower, more careful boosting), depth decreased from 6 to 4 (reducing overfitting), and n_estimators increased from 200 to 300 (more rounds to compensate for the lower learning rate). Unlike Random Forest, XGBoost improved both AUC and recall (from 0.509 to 0.638), going from worst of three at baseline to best after tuning.

### 5.5 Tuning Progress Log

In [ ]:
tuning_log = pd.DataFrame([
    {'Model': 'Logistic Regression', 'Stage': 'Baseline', 'AUC': 0.6982},
    {'Model': 'Logistic Regression', 'Stage': 'RandomizedSearch', 'AUC': lr_random.best_score_},
    {'Model': 'Logistic Regression', 'Stage': 'GridSearch', 'AUC': lr_grid.best_score_},
    {'Model': 'Random Forest', 'Stage': 'Baseline', 'AUC': 0.6961},
    {'Model': 'Random Forest', 'Stage': 'RandomizedSearch', 'AUC': rf_random.best_score_},
    {'Model': 'Random Forest', 'Stage': 'GridSearch', 'AUC': rf_grid.best_score_},
    {'Model': 'XGBoost', 'Stage': 'Baseline', 'AUC': 0.6788},
    {'Model': 'XGBoost', 'Stage': 'RandomizedSearch', 'AUC': xgb_random.best_score_},
    {'Model': 'XGBoost', 'Stage': 'GridSearch', 'AUC': xgb_grid.best_score_},
    {'Model': 'SVM', 'Stage': 'Baseline (excluded)', 'AUC': 0.6754},
])

print("TUNING PROGRESS LOG")
print("=" * 60)
print(tuning_log.to_string(index=False))

In [ ]:
# Tuning progress visualization
fig, ax = plt.subplots(figsize=(12, 5))

models_to_plot = ['Logistic Regression', 'Random Forest', 'XGBoost']
stages = ['Baseline', 'RandomizedSearch', 'GridSearch']
color_map = {
    'Logistic Regression': COLORS['steel'],
    'Random Forest': COLORS['teal'],
    'XGBoost': COLORS['orange'],
}

x = np.arange(len(stages))
width = 0.25

for i, model in enumerate(models_to_plot):
    model_data = tuning_log[tuning_log['Model'] == model]
    aucs = [model_data[model_data['Stage'] == s]['AUC'].values[0] for s in stages]
    bars = ax.bar(x + i * width, aucs, width, label=model, color=color_map[model], edgecolor='white')
    for bar, auc in zip(bars, aucs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
                f'{auc:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xlabel('Tuning Stage', fontsize=12)
ax.set_ylabel('ROC-AUC (5-Fold CV)', fontsize=12)
ax.set_title('Hyperparameter Tuning Progress by Model', fontsize=14, fontweight='bold', color=COLORS['primary'])
ax.set_xticks(x + width)
ax.set_xticklabels(stages)
ax.legend(loc='lower right')
ax.set_ylim(0.65, max(tuning_log['AUC']) + 0.015)
ax.axhline(y=0.65, color=COLORS['red'], linestyle='--', alpha=0.5, label='Success threshold')

plt.tight_layout()
plt.savefig('../figures/tuning_progress.png', dpi=150, bbox_inches='tight')
plt.show()

**Tuning progress:** XGBoost showed the largest improvement, going from worst baseline (0.679) to best tuned (0.703) -- a +0.024 gain. Logistic Regression was nearly flat (0.698 to 0.699, +0.001), confirming the baseline was already well-configured. Random Forest improved modestly on AUC (0.696 to 0.701, +0.005) but at the severe cost of recall dropping from 0.574 to 0.224. The model ranking changed from LR > RF > XGB at baseline to XGB > RF > LR after tuning, validating the decision to include XGBoost despite its 3rd-place baseline finish.

### 5.6 Final Model Comparison (Tuned)

In [ ]:
from sklearn.model_selection import cross_validate

# Final CV comparison of tuned models
tuned_models = {
    'LR (tuned)': lr_grid.best_estimator_,
    'RF (tuned)': rf_grid.best_estimator_,
    'XGB (tuned)': xgb_grid.best_estimator_,
}

tuned_results = {}
for name, pipeline in tuned_models.items():
    cv_res = cross_validate(pipeline, X_train, y_train,
                            cv=cv, scoring=scoring, n_jobs=-1)
    tuned_results[name] = {
        'auc': cv_res['test_roc_auc'].mean(),
        'recall': cv_res['test_recall'].mean(),
        'precision': cv_res['test_precision'].mean(),
        'f1': cv_res['test_f1'].mean(),
    }

# Print comparison table: baseline vs tuned
print("\nBASELINE vs TUNED COMPARISON")
print("=" * 75)
print(f"{'Model':<20} {'Baseline AUC':>14} {'Tuned AUC':>12} {'Delta':>8} {'Tuned Recall':>14}")
print("-" * 75)
comparisons = [
    ('LR', 0.6982, tuned_results['LR (tuned)']),
    ('RF', 0.6961, tuned_results['RF (tuned)']),
    ('XGB', 0.6788, tuned_results['XGB (tuned)']),
]
for name, baseline_auc, tuned in comparisons:
    delta = tuned['auc'] - baseline_auc
    print(f"{name:<20} {baseline_auc:>14.4f} {tuned['auc']:>12.4f} {delta:>+8.4f} {tuned['recall']:>14.4f}")

In [ ]:
# Baseline vs Tuned AUC comparison chart
fig, ax = plt.subplots(figsize=(14, 5))

model_labels = ['Logistic Regression', 'Random Forest', 'XGBoost']
baseline_aucs = [0.6982, 0.6961, 0.6788]
tuned_aucs = [
    tuned_results['LR (tuned)']['auc'],
    tuned_results['RF (tuned)']['auc'],
    tuned_results['XGB (tuned)']['auc'],
]

x = np.arange(len(model_labels))
width = 0.3

bars1 = ax.bar(x - width/2, baseline_aucs, width, label='Baseline',
               color=COLORS['steel'], alpha=0.6, edgecolor='white')
bars2 = ax.bar(x + width/2, tuned_aucs, width, label='Tuned',
               color=COLORS['teal'], edgecolor='white')

# Add value labels
for bar, auc in zip(bars1, baseline_aucs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
            f'{auc:.4f}', ha='center', va='bottom', fontsize=10, color=COLORS['steel'])
for bar, auc in zip(bars2, tuned_aucs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
            f'{auc:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold', color=COLORS['teal'])

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('ROC-AUC (5-Fold CV)', fontsize=12)
ax.set_title('Baseline vs Tuned Model Performance', fontsize=14, fontweight='bold', color=COLORS['primary'])
ax.set_xticks(x)
ax.set_xticklabels(model_labels)
ax.legend()
ax.set_ylim(0.65, max(max(baseline_aucs), max(tuned_aucs)) + 0.015)
ax.axhline(y=0.65, color=COLORS['red'], linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../figures/baseline_vs_tuned.png', dpi=150, bbox_inches='tight')
plt.show()

**Baseline vs tuned comparison:** All three models improved on AUC after tuning: LR +0.001 (0.6982 to 0.6992), RF +0.005 (0.6961 to 0.7009), XGB +0.024 (0.6788 to 0.7025). However, the recall story diverged sharply -- RF recall regressed from 0.574 to 0.224 while XGBoost recall improved from 0.509 to 0.638. XGBoost was the only model that improved on both AUC and recall after tuning, making it the clear choice for the final model. The RF recall collapse reinforces a key lesson: AUC measures ranking quality across all thresholds, but for imbalanced clinical classification, recall at the operating threshold matters just as much.

### 5.7 Select Final Model

In [ ]:
import joblib

# Select final model based on best tuned AUC
final_model_name = max(tuned_results, key=lambda k: tuned_results[k]['auc'])
final_pipeline = tuned_models[final_model_name]

# Fit on full training set
final_pipeline.fit(X_train, y_train)

# Save
joblib.dump(final_pipeline, '../models/final_model.joblib')

print(f"FINAL MODEL SELECTED: {final_model_name}")
print(f"  CV AUC: {tuned_results[final_model_name]['auc']:.4f}")
print(f"  CV Recall: {tuned_results[final_model_name]['recall']:.4f}")
print(f"  CV Precision: {tuned_results[final_model_name]['precision']:.4f}")
print(f"  CV F1: {tuned_results[final_model_name]['f1']:.4f}")
print(f"\nBest hyperparameters:")
for param, value in final_pipeline.named_steps['classifier'].get_params().items():
    if param in ['C', 'penalty', 'n_estimators', 'max_depth', 'learning_rate',
                 'min_samples_leaf', 'max_features', 'subsample', 'colsample_bytree',
                 'min_child_weight', 'scale_pos_weight', 'class_weight']:
        print(f"  {param}: {value}")

print(f"\nModel saved to ./models/final_model.joblib")
print(f"Test set remains LOCKED until Phase 6.")

**Model selection rationale:** XGBoost was selected as the final model because it achieved the highest tuned AUC (0.7025), the highest tuned recall (0.638), and the fastest total tuning time (~2 minutes). Key hyperparameters: learning_rate=0.03, max_depth=4, n_estimators=300, subsample=0.8, colsample_bytree=0.7, scale_pos_weight=13.27. XGBoost is also the published benchmark model for this dataset (Strack et al., 2014), so the result aligns with the literature. The model has been fit on the full training set and saved to disk. The test set remains completely untouched -- it will be used only once in Phase 6 for final evaluation, ensuring an unbiased estimate of generalization performance.

### Phase 5 Complete

**Tuning results:**
- [x] Logistic Regression tuned: Baseline AUC 0.6982 -> Tuned AUC 0.6992 (+0.001)
- [x] Random Forest tuned: Baseline AUC 0.6961 -> Tuned AUC 0.7009 (+0.005, but recall dropped from 0.574 to 0.224)
- [x] XGBoost tuned: Baseline AUC 0.6788 -> Tuned AUC 0.7025 (+0.024, recall improved from 0.509 to 0.638)
- [x] SVM excluded (45-min training time, worst baseline AUC)
- [x] Iterative improvement documented in tuning progress log
- [x] Final model selected: XGBoost (tuned) -- AUC 0.7025, Recall 0.638
- [x] Final model saved to ./models/final_model.joblib
- [x] Test set remains LOCKED

**Next: Phase 6 -- Final Evaluation, SHAP Analysis, and Business Impact**

### Save Phase 5 Artifacts

In [ ]:
# Save Phase 5 artifacts for Phase 6

# Save all tuned models
for name, pipeline in tuned_models.items():
    safe_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    joblib.dump(pipeline, f'{ARTIFACTS_DIR}/phase5_tuned_{safe_name}.joblib')
    print(f'  Saved: phase5_tuned_{safe_name}.joblib')

# Save final model
joblib.dump(final_pipeline, f'{ARTIFACTS_DIR}/phase5_best_model.joblib')
joblib.dump(final_pipeline, '../models/final_model.joblib')

# Save Phase 5 metadata
final_name = max(tuned_results, key=lambda k: tuned_results[k]['auc'])
phase5_meta = {
    'best_model_name': final_name,
    'best_cv_roc_auc': float(tuned_results[final_name]['auc']),
    'best_cv_recall': float(tuned_results[final_name]['recall']),
    'tuning_results': {name: dict(r) for name, r in tuned_results.items()},
    'tuned_model_names': list(tuned_models.keys()),
}
with open(f'{ARTIFACTS_DIR}/phase5_meta.pkl', 'wb') as f:
    pickle.dump(phase5_meta, f)

print(f'\nPhase 5 artifacts saved')
print(f'  phase5_best_model.joblib')
print(f'  phase5_meta.pkl')
print(f'  ../models/final_model.joblib')